In [ ]:
import numpy as np
#import matplotlib
#%matplotlib notebook
import matplotlib.pyplot as plt
import pylcp
from pylcp.fields import gaussianBeam
from pylcp.common import cart2spherical
import cProfile, pstats, io

In [ ]:
# 50 gauss/cm for Sr 
# mub on database
# 0.43 cm for x0 30/1.4/50

# x is length unit
# k is wavenumber which relates to wavelength
x0 = (30/1.4/50) # cm
k = 2*np.pi/461E-7 # cm^{-1}
kbar = k*x0

# D source for position
d = np.array([0., -8.839/10, -8.839/10])/x0

# Gamma is decay rate
# t0 is normalized time units of decary
# wb is width factor
gamma = 2*np.pi*30e6
t0 = 1e-5 # s
gammabar = gamma*t0
wb = 4.7 # mm

In [ ]:
class clippedGaussianBeam(pylcp.laserBeam):
    def __init__(self, kvec, pol, beta, delta, r0=np.array([0.,0.,0.]), wdiv=1, w0=1, wcut=1000, **kwargs):
        if callable(kvec):
            raise TypeError('kvec cannot be a function for a Gaussian beam.')

        if callable(pol):
            raise TypeError('Polarization cannot be a function for a Gaussian beam.')

        # Save the constant values (might be useful):
        self.con_kvec = kvec
        self.con_kmag = np.linalg.norm(kvec)
        self.con_khat = kvec/self.con_kmag
        if isinstance(pol, int) or isinstance(pol, float):
            if pol>0:
                pol = np.array([1., -1j, 0.])/np.sqrt(2)
            else:
                pol = np.array([1., +1j, 0.])/np.sqrt(2)
        self.con_pol = pol

        # Save the parameters specific to the Gaussian beam:
        self.beta_max = beta    # central saturation parameter
        self.wavelength = 2*np.pi/(np.linalg.norm(kvec))  
        self.r0 = r0    # Position of focus
        self.wdiv = wdiv
        self.w0 = w0
        self.wcut = wcut
        self.wcutfract = self.wcut/self.w0
        self.zr = np.pi*self.wdiv**2/(self.wavelength)  # Rayleigh length

        # Define the global rotation matrix
        self.global_rotation_matrix()
        
        # Use super class to define delta(t):
        super().__init__(delta=delta, **kwargs)
   

    def global_rotation_matrix(self):
        th = np.arccos(self.con_khat[2])
        ph = np.arctan2(self.con_khat[1], self.con_khat[0])
        
        rz = np.array([[np.cos(ph),-np.sin(ph),0.], [np.sin(ph),np.cos(ph),0.], [0.,0.,1.]])
        ry = np.array([[np.cos(th),0.,np.sin(th)], [0.,1.,0.], [-np.sin(th),0.,np.cos(th)]])
        
        self.rmat = rz@ry@np.linalg.inv(rz)
        self.rmat_inv = np.linalg.inv(self.rmat)
        return self.rmat
    
    
    def define_rotation_matrix(self):
        # Angles of rotation:|
        th = np.arccos(self.con_khat[2])
        phi = np.arctan2(self.con_khat[1], self.con_khat[0])
        
        # Use scipy to define the rotation matrix
        self.rmat = Rotation.from_euler('ZY', [phi, th]).inv().as_matrix() 
        self.rmat_inv = np.linalg.inv(self.rmat)

        
    def local_parameters(self, R=np.array([0., 0., 0.]), t=0.):
        # Adjust offset:
        Rp = R - self.r0.reshape((3,) + (1,)*(R.ndim-1))
    
        # Rotate up to the z-axis where we can apply formulas:
        Rp = np.einsum('ij,j...->i...', self.rmat_inv, Rp)
        rho_sq=np.sum(Rp[:2]**2, axis=0)
       
        # The waist at the position of interest:
        w = self.w0*np.sqrt(1+(Rp[2]**2/self.zr**2)) # w0*sqrt(1+(z/zr).^2);
        
        # Return the intensity:
        I = self.beta_max*(self.w0**2)/(w**2)*np.exp(-2*rho_sq/w**2)*(np.sqrt(rho_sq)<w*self.wcutfract) #Beta*(w0./w).^2.*exp(-2*r.^2./w.^2).*(r<w*wcutfract);
        
        # Now calculate the local k-vector in cylindrical coordinates:
        kr = (np.sqrt(rho_sq))*(Rp[2])/(self.zr**2+Rp[2]**2) # r.*z./(zr^2+z.^2);
        kt = np.zeros(Rp[0].shape)
        kz = np.ones(Rp[0].shape)
        
        # Convert to Cartesian: 
        kx = (kr*Rp[0]+kt*Rp[1])/(np.sqrt(Rp[0]**2+Rp[1]**2+1e-100))
        ky = (kr*Rp[1]+kt*Rp[0])/(np.sqrt(Rp[0]**2+Rp[1]**2+1e-100))
        
        # Normalize:
        kxn=kx/np.sqrt(kx**2+ky**2+kz**2) # normalized
        kyn=ky/np.sqrt(kx**2+ky**2+kz**2)
        kzn=kz/np.sqrt(kx**2+ky**2+kz**2)
        
        # Put into full array:
        kn=np.array([kxn,kyn,kzn])
        
        # Think about a way to do this without having to this without the FOR loop:
        it = np.nditer([kn[0], kn[1], kn[2], None, None, None], op_dtypes=['float64', 'float64', 'float64', 'complex128', 'complex128', 'complex128'])
        for (kxn, kyn, kzn, px, py, pz) in it:       
            thn = np.arccos(kzn)
            phn = np.arctan2(kyn, kxn)
            
            rzn = np.array([[np.cos(phn), -np.sin(phn), 0.],
                            [np.sin(phn),  np.cos(phn), 0.],
                            [         0.,           0., 1.]])
            ryn = np.array([[np.cos(thn),  0., np.sin(thn)],
                            [0.,           1.,          0.],
                            [-np.sin(thn), 0., np.cos(thn)]])
            
            rmatn = rzn@ryn@np.linalg.inv(rzn)

            (px[...], py[...], pz[...]) = self.rmat@rmatn@np.transpose(self.con_pol)
            
        # Rotate back:
        k = np.einsum('ij,j...->i...', self.rmat, kn)*self.con_kmag
        
        return k, cart2spherical(np.array(it.operands[3:])), I, self.delta(t)
    
    
    def beta(self, R=np.array([0., 0., 0.]), t=0.):
        k, P, I, D = self.local_parameters(R, t)
        return I
    
    def pol(self, R=np.array([0., 0., 0.]), t=0.):
        k, P, I, D = self.local_parameters(R, t)
        return P
    
    def kvec(self, R=np.array([0., 0., 0.]), t=0.):
        k, P, I, D = self.local_parameters(R, t)
        return k

# Set up the laser beams with their appropriate characteristics
delta = -1.5*gammabar
beta = 10
sigma = 1.0

In [ ]:
nr=3
        
class clippedGauss(pylcp.laserBeams):
     def __init__(self, *args, **kwargs):
        super().__init__()
    
        beam_type = kwargs.pop('beam_type', pylcp.laserBeam)
        pol = kwargs.pop('pol', +1)
        kmag = kwargs.pop('k', 1.)
        beta = kwargs.pop('beta', 1.0)
        delta = kwargs.pop('delta', -1.5*gammabar)
        kmag = 2*np.pi/461e-6
        
        self.add_laser(clippedGaussianBeam(kvec=kmag*np.array([0., 0., -1.]), \
                       pol=np.array([1., 1.j, 0])/np.sqrt(2), beta=1.0*beta, delta=delta, r0=np.array([0., 0., 0.]), wdiv=4.65, w0=4.65*100, wcut=4.65, **kwargs))
        self.add_laser(clippedGaussianBeam(kvec=kmag*np.array([0.612, 0.75, -0.25]), \
                       pol=np.array([1., -1.j, 0])/np.sqrt(2), beta=1.0*beta, delta=delta, r0=np.array([0., 0., 0.]), wdiv=4.65, w0=4.65*100, wcut=4.65, **kwargs))
        self.add_laser(clippedGaussianBeam(kvec=kmag*np.array([-0.612, 0.75, -0.25]), \
                       pol=np.array([1., -1.j, 0])/np.sqrt(2), beta=1.0*beta, delta=delta, r0=np.array([0., 0., 0.]), wdiv=4.65, w0=4.65*100, wcut=4.65, **kwargs))
        self.add_laser(clippedGaussianBeam(kvec=kmag*np.array([0., 0., 1.]), \
                      pol=np.array([1., 1.j, 0])/np.sqrt(2), beta=6.261e7*beta, delta=delta, r0=np.array([0., 0., -17.788]), wdiv=5.744e-4, w0=5.744e-4*100, wcut=5.744e-4, **kwargs))
        self.add_laser(clippedGaussianBeam(kvec=kmag*np.array([-0.612, -0.75, 0.25]), \
                       pol=np.array([1., -1.j, 0])/np.sqrt(2), beta=6.261e7*beta, delta=delta, r0=np.array([10.893, 13.341, -4.447]), wdiv=5.744e-4, w0=5.744e-4*100, wcut=5.744e-4, **kwargs))
        self.add_laser(clippedGaussianBeam(kvec=kmag*np.array([0.612, -0.75, 0.25]), \
                       pol=np.array([1., -1.j, 0])/np.sqrt(2), beta=6.261e7*beta, delta=delta, r0=np.array([-10.893, 13.341, -4.447]), wdiv=5.744e-4, w0=5.744e-4*100, wcut=5.744e-4, **kwargs))
    
# Set up the laser beams with their appropriate characteristics
det = -1.5*gammabar
beta = 1.0

#beam_to_sim = pylcp.infinitePlaneWaveBeam
beam_to_sim = pylcp.gaussianBeam
#beam_to_sim = pylcp.clippedGaussianBeam

#MOT_to_sim = conventional3DMOTBeams
#MOT_to_sim = aPHIMOT
MOT_to_sim = clippedGauss

#MOT_to_sim_kwargs = {'rotation_angles':[np.pi/4, 0., 0.]} # Extra arguments for conventional3DMOTBeams
MOT_to_sim_kwargs = {} # Extra arguments for aPHIMOT/rotDivGauss

laser_kwargs = {} # Use for rotDivGauss
#laser_kwargs = {'wb':wb} # Extra arguments for GaussianBeam
#laser_kwargs = {'wb':1000*wb, 'rs':wb} # Extra arguments for clippedGaussianBeam

testBeams = MOT_to_sim(beta=beta, delta=det, beam_type=beam_to_sim, k=kbar, **laser_kwargs, **MOT_to_sim_kwargs) # kbar

############################################################################################

#Trigger numba to compile the beta code:
testBeams.beam_vector[0].beta()
testBeams.beam_vector[1].beta()

x_beta = 15
X, Y = np.meshgrid(np.linspace(-x_beta, x_beta, 101),
                   np.linspace(-x_beta, x_beta, 101))
z_tests = [-4, 0, +4]

plt.figure("Laser Beams", figsize=(4, 1.5*nr))
plt.clf()
pr = cProfile.Profile()

for jj, laserBeam in enumerate(testBeams.beam_vector):
    for ii, z_test in enumerate(z_tests):
        Z = z_test*np.ones(X.shape)
        it = np.nditer([X, Y, Z, None])
        Rt=np.array([X, Y, Z])

        """pr.enable()
        for (x, y, z, beta) in it:
            beta[...] = laserBeam.beta(np.array([x, y, z]), 0.)
        pr.disable()
        
        BETA = it.operands[3]"""
        
        pr.enable()
        BETA = laserBeam.beta(Rt)
        pr.disable()
        
        plt.subplot(len(testBeams.beam_vector), len(z_tests), jj*len(z_tests)+ii+1)
        plt.imshow(BETA, origin='lower',
                   extent=(-x_beta, x_beta,
                           -x_beta, x_beta))
        plt.clim((0, 2))
        plt.set_cmap('jet')
        # Make a cross-hair:
        plt.plot([0, 0], [-x_beta, x_beta],
                 'w-', linewidth=0.25)
        plt.plot([-x_beta, x_beta], [0, 0],
                 'w-', linewidth=0.25)

In [ ]:
zchip = 1.0
Z = (zchip-8)*np.ones(X.shape)

Rt=np.array([X, Y, Z])
plt.imshow(np.sum(testBeams.beta(Rt)[1::], axis=0))

In [ ]:
# s = io.StringIO()
# sortby = 'cumtime'
# ps = pstats.Stats(pr, stream=s).sort_stats(sortby)
# ps.print_stats()
# print(s.getvalue())

In [ ]:
nr=3
# testBeams = gratings.infiniteGratingMOTBeams(delta=-1., s=1., nr=nr, thd=np.pi/4,
#                                              pol=np.array([-1/np.sqrt(2), 1j/np.sqrt(2), 0]),
#                                              reflected_pol=np.array([np.pi, 0]),
#                                              reflected_pol_basis='poincare', eta=None, grating_angle=0)
x_beta = 2
X, Y = np.meshgrid(np.linspace(-x_beta, x_beta, 101),
                   np.linspace(-x_beta, x_beta, 101))
z_tests = [0]

plt.figure("Infinite Beams", figsize=(4, 1.5*nr))
plt.clf()
for jj, laserBeam in enumerate(testBeams.beam_vector):
    for ii, z_test in enumerate(z_tests):
        plt.subplot(len(testBeams.beam_vector), len(z_tests), jj*len(z_tests)+ii+1)
        Rt=np.array([X, Y, z_test*np.ones(X.shape)])
        tt=np.zeros(X.shape)
        print(laserBeam.kvec())
        plt.imshow(laserBeam.beta(R=Rt,t=tt),
                   origin='lower',
                   extent=(-x_beta, x_beta,
                           -x_beta, x_beta))
        plt.clim((0, 1))
        plt.set_cmap('jet')
        # Make a cross-hair:
        plt.plot([0, 0], [-x_beta, x_beta],
                 'w-', linewidth=0.25)
        plt.plot([-x_beta, x_beta], [0, 0],
                 'w-', linewidth=0.25)